# Day 05 — Model Optimization
## Sampling Strategies × Threshold Tuning × TimeSeriesSplit CV × Optuna

**Steps:**
1. Ingestion + Pipeline (from scratch)
2. Compare 7 sampling strategies
3. TimeSeriesSplit cross-validation
4. Selected strategy: **RandomOverSampler** — hyperparameter optimization with Optuna

### Split design
```
Gold (sorted by time_6h)
├── Train  80%
│   ├── Fit set   first 85%  ← sampling + model training
│   └── Val set   last  15%  ← threshold tuning (raw, no resample)
└── Test   20%  ────────────── final evaluation (never touched)

CV (Optuna): TimeSeriesSplit(n_splits=5), ROS applied inside each training fold
```

---
## 0. Setup

In [25]:
import sys, warnings
sys.path.insert(0, "..")
warnings.filterwarnings("ignore")

import json
import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.model_selection import TimeSeriesSplit, cross_val_score
from sklearn.metrics import (
    roc_auc_score, average_precision_score, f1_score, fbeta_score,
    precision_score, recall_score, precision_recall_curve,
    classification_report,
)
from xgboost import XGBClassifier
import joblib

print("Core libraries ready.")

Core libraries ready.


In [26]:
# imbalanced-learn
try:
    import imblearn
    print(f"imbalanced-learn {imblearn.__version__}")
except ImportError:
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "imbalanced-learn", "-q"])
    import imblearn
    print(f"Installed imbalanced-learn {imblearn.__version__}")

from imblearn.over_sampling  import RandomOverSampler, SMOTE, ADASYN
from imblearn.under_sampling import RandomUnderSampler, TomekLinks
from imblearn.combine        import SMOTETomek
from imblearn.pipeline       import Pipeline as ImbPipeline

# optuna
try:
    import optuna
    print(f"optuna {optuna.__version__}")
except ImportError:
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "optuna", "-q"])
    import optuna
    print(f"Installed optuna {optuna.__version__}")

print("All libraries ready.")

imbalanced-learn 0.14.1
optuna 4.8.0
All libraries ready.


---
## 1. Ingestion & Pipeline

In [27]:
from src import ingestion, config

print(f"Date range : {config.HISTORICAL_START}  →  {config.HISTORICAL_END}")
print(f"Zones      : {[z['zone'] for z in config.BAKU_ZONES]}")
print()
print("Running historical ingestion (Bronze)...")
ingestion.run_historical_ingest()
print("Done.")

Date range : 2020-01-01  →  2026-04-20
Zones      : ['High Relief', 'Low Relief', 'Moderate Relief']

Running historical ingestion (Bronze)...
Done.


In [28]:
from src import pipeline

print("Building Silver → Gold...")
pipeline.run_pipeline()
print("Done.")

df = pipeline.load_gold()
print(f"\nGold: {df.shape[0]:,} rows  x  {df.shape[1]} columns")
df.head(3)

Building Silver → Gold...
Done.

Gold: 27,624 rows  x  39 columns


,zone,time_6h,temperature_2m,relative_humidity_2m,precipitation,wind_speed_10m,soil_moisture_0_to_7cm,soil_moisture_7_to_28cm,soil_temperature_0_to_7cm,et0_fao_evapotranspiration,...,humidity_precip_product,highland_precip_24h,zone_cascade_risk,hour_sin,hour_cos,doy_sin,doy_cos,is_winter,dq_score,is_flood
0,Moderate Relief,2025-08-10 00:00:00,25.050000,77.500000,0.0,24.866667,0.022667,0.136,26.083333,0.43,...,0.0,0.0,0.0,0.000000e+00,1.000000e+00,-0.628763,-0.777597,0,1.0,0
1,Moderate Relief,2025-08-10 06:00:00,27.233333,65.333333,0.0,22.683333,0.026000,0.136,27.566667,1.66,...,0.0,0.0,0.0,1.000000e+00,6.123234e-17,-0.628763,-0.777597,0,1.0,0
2,Moderate Relief,2025-08-10 12:00:00,30.500000,43.000000,0.0,22.916667,0.026667,0.136,36.033333,3.37,...,0.0,0.0,0.0,1.224647e-16,-1.000000e+00,-0.628763,-0.777597,0,1.0,0


---
## 2. Class Imbalance

In [29]:
TARGET     = "is_flood"
counts     = df[TARGET].value_counts().sort_index()
flood_rate = df[TARGET].mean()

print(f"Total rows     : {len(df):,}")
print(f"Flood events   : {counts.get(1, 0):,}   ({flood_rate*100:.2f}%)")
print(f"Non-flood      : {counts.get(0, 0):,}  ({(1-flood_rate)*100:.2f}%)")
print(f"Imbalance ratio: 1 : {(1-flood_rate)/max(flood_rate, 1e-9):.0f}")
print("\nFlood rate by zone:")
print((df.groupby("zone")[TARGET].mean() * 100).round(2).to_string())

Total rows     : 27,624
Flood events   : 293   (1.06%)
Non-flood      : 27,331  (98.94%)
Imbalance ratio: 1 : 93

Flood rate by zone:
zone
High Relief        0.40
Low Relief         2.42
Moderate Relief    0.36


---
## 3. Feature Preparation & Splits

In [30]:
from src.model import prepare_features, _time_split

train_df, test_df = _time_split(df, test_frac=0.20)

X_train_raw, feature_cols = prepare_features(train_df)
X_test_raw,  _            = prepare_features(test_df)
for c in feature_cols:
    if c not in X_test_raw.columns:
        X_test_raw[c] = 0
X_test_raw = X_test_raw[feature_cols]

y_train = train_df[TARGET].values
y_test  = test_df[TARGET].values
X_train = X_train_raw.fillna(0).values
X_test  = X_test_raw.fillna(0).values

print(f"Features : {len(feature_cols)}")
print(f"Train    : {X_train.shape}   flood rate = {y_train.mean()*100:.2f}%")
print(f"Test     : {X_test.shape}    flood rate = {y_test.mean()*100:.2f}%")

Features : 37
Train    : (22098, 37)   flood rate = 0.62%
Test     : (5526, 37)    flood rate = 2.82%


In [31]:
# Validation window carved from the end of train — never resampled, used only for threshold tuning
val_cut = int(len(X_train) * 0.85)
X_tr,  y_tr  = X_train[:val_cut], y_train[:val_cut]
X_val, y_val = X_train[val_cut:], y_train[val_cut:]

print(f"Fit set  : {X_tr.shape}   flood rate = {y_tr.mean()*100:.2f}%")
print(f"Val set  : {X_val.shape}  flood rate = {y_val.mean()*100:.2f}%")
print(f"Test set : {X_test.shape}  flood rate = {y_test.mean()*100:.2f}%")

Fit set  : (18783, 37)   flood rate = 0.65%
Val set  : (3315, 37)  flood rate = 0.45%
Test set : (5526, 37)  flood rate = 2.82%


---
## 4. Helper Functions

In [32]:
# scale_pos_weight removed — each sampling strategy handles imbalance independently
BASE_XGB_PARAMS = dict(
    n_estimators     = 500,
    learning_rate    = 0.05,
    max_depth        = 5,
    min_child_weight = 3,
    subsample        = 0.8,
    colsample_bytree = 0.8,
    reg_alpha        = 0.1,
    reg_lambda       = 1.0,
    eval_metric      = "aucpr",
    random_state     = 42,
    n_jobs           = -1,
)

# Recall-biased threshold: maximize F2-score (beta=2 weights recall 4x over precision)
# min_precision floor prevents the threshold from collapsing to 0 and predicting everything as flood
def best_threshold_for_recall(model, X, y, beta=2.0, min_precision=0.05):
    probs = model.predict_proba(X)[:, 1]
    precision, recall, thresholds = precision_recall_curve(y, probs)
    fbeta = ((1 + beta**2) * precision * recall
             / (beta**2 * precision + recall + 1e-9))
    # Enforce minimum precision floor on the threshold candidates
    valid = precision[:-1] >= min_precision
    if valid.any():
        masked = np.where(valid, fbeta[:-1], -1.0)
        return float(thresholds[np.argmax(masked)])
    # Fallback: no threshold meets min_precision — take best F2 anyway
    return float(thresholds[np.argmax(fbeta[:-1])])


def evaluate(model, X, y, threshold, name=""):
    probs = model.predict_proba(X)[:, 1]
    preds = (probs >= threshold).astype(int)
    return {
        "Strategy" : name,
        "AUC-ROC"  : round(roc_auc_score(y, probs), 4),
        "AUC-PR"   : round(average_precision_score(y, probs), 4),
        "F1"       : round(f1_score(y, preds, zero_division=0), 4),
        "F2"       : round(fbeta_score(y, preds, beta=2, zero_division=0), 4),
        "Precision": round(precision_score(y, preds, zero_division=0), 4),
        "Recall"   : round(recall_score(y, preds, zero_division=0), 4),
        "Threshold": round(threshold, 4),
    }, probs


def run_strategy(name, sampler, params=None):
    """Resample fit set, train XGB, tune threshold for recall (F2), evaluate on test."""
    xgb_params = params if params else BASE_XGB_PARAMS
    if sampler is not None:
        X_s, y_s = sampler.fit_resample(X_tr, y_tr)
    else:
        X_s, y_s = X_tr, y_tr

    print(f"\n{'─'*58}")
    print(f"  {name}")
    print(f"  Fit rows : {X_s.shape[0]:,}   flood rate = {y_s.mean()*100:.1f}%")

    mdl = XGBClassifier(**xgb_params)
    mdl.fit(X_s, y_s, eval_set=[(X_val, y_val)], verbose=False)

    threshold      = best_threshold_for_recall(mdl, X_val, y_val)
    metrics, probs = evaluate(mdl, X_test, y_test, threshold, name)

    print(f"  Threshold (F2-tuned) : {threshold:.4f}")
    print(f"  AUC-ROC  = {metrics['AUC-ROC']:.4f}")
    print(f"  AUC-PR   = {metrics['AUC-PR']:.4f}")
    print(f"  F2       = {metrics['F2']:.4f}   "
          f"F1 = {metrics['F1']:.4f}   "
          f"Precision = {metrics['Precision']:.4f}   "
          f"Recall = {metrics['Recall']:.4f}")
    return mdl, threshold, metrics, probs


all_results  = {}
all_probs    = {}
saved_models = {}
print("Helpers ready.")

Helpers ready.


---
## 5. Strategy Comparison
Sampling applied **only** to the fit set. Val and test sets are always raw.

In [33]:
m, t, metrics, probs = run_strategy("Baseline", None)
all_results["Baseline"]  = metrics
all_probs["Baseline"]    = probs
saved_models["Baseline"] = (m, t)


──────────────────────────────────────────────────────────
  Baseline
  Fit rows : 18,783   flood rate = 0.6%
  Threshold (F2-tuned) : 0.2190
  AUC-ROC  = 0.9314
  AUC-PR   = 0.4004
  F2       = 0.2908   F1 = 0.3460   Precision = 0.5062   Recall = 0.2628


In [34]:
m, t, metrics, probs = run_strategy("RandomOverSampler", RandomOverSampler(random_state=42))
all_results["RandomOverSampler"]  = metrics
all_probs["RandomOverSampler"]    = probs
saved_models["RandomOverSampler"] = (m, t)


──────────────────────────────────────────────────────────
  RandomOverSampler
  Fit rows : 37,322   flood rate = 50.0%
  Threshold (F2-tuned) : 0.1739
  AUC-ROC  = 0.9231
  AUC-PR   = 0.4480
  F2       = 0.4883   F1 = 0.4633   Precision = 0.4270   Recall = 0.5064


In [35]:
m, t, metrics, probs = run_strategy("RandomUnderSampler", RandomUnderSampler(random_state=42))
all_results["RandomUnderSampler"]  = metrics
all_probs["RandomUnderSampler"]    = probs
saved_models["RandomUnderSampler"] = (m, t)


──────────────────────────────────────────────────────────
  RandomUnderSampler
  Fit rows : 244   flood rate = 50.0%
  Threshold (F2-tuned) : 0.9670
  AUC-ROC  = 0.9128
  AUC-PR   = 0.3106
  F2       = 0.1844   F1 = 0.2381   Precision = 0.4630   Recall = 0.1603


In [36]:
m, t, metrics, probs = run_strategy("SMOTE", SMOTE(random_state=42, k_neighbors=5))
all_results["SMOTE"]  = metrics
all_probs["SMOTE"]    = probs
saved_models["SMOTE"] = (m, t)


──────────────────────────────────────────────────────────
  SMOTE
  Fit rows : 37,322   flood rate = 50.0%
  Threshold (F2-tuned) : 0.5657
  AUC-ROC  = 0.9238
  AUC-PR   = 0.3617
  F2       = 0.1920   F1 = 0.2298   Precision = 0.3418   Recall = 0.1731


In [37]:
try:
    m, t, metrics, probs = run_strategy("ADASYN", ADASYN(random_state=42))
    all_results["ADASYN"]  = metrics
    all_probs["ADASYN"]    = probs
    saved_models["ADASYN"] = (m, t)
except Exception as exc:
    print(f"\nADASYN skipped — {exc}")


──────────────────────────────────────────────────────────
  ADASYN
  Fit rows : 37,347   flood rate = 50.0%
  Threshold (F2-tuned) : 0.6730
  AUC-ROC  = 0.9271
  AUC-PR   = 0.3637
  F2       = 0.1968   F1 = 0.2477   Precision = 0.4355   Recall = 0.1731


In [38]:
m, t, metrics, probs = run_strategy("TomekLinks", TomekLinks(n_jobs=-1))
all_results["TomekLinks"]  = metrics
all_probs["TomekLinks"]    = probs
saved_models["TomekLinks"] = (m, t)


──────────────────────────────────────────────────────────
  TomekLinks
  Fit rows : 18,732   flood rate = 0.7%
  Threshold (F2-tuned) : 0.0494
  AUC-ROC  = 0.9329
  AUC-PR   = 0.4228
  F2       = 0.4774   F1 = 0.4470   Precision = 0.4041   Recall = 0.5000


In [39]:
m, t, metrics, probs = run_strategy("SMOTETomek", SMOTETomek(random_state=42))
all_results["SMOTETomek"]  = metrics
all_probs["SMOTETomek"]    = probs
saved_models["SMOTETomek"] = (m, t)


──────────────────────────────────────────────────────────
  SMOTETomek
  Fit rows : 37,314   flood rate = 50.0%
  Threshold (F2-tuned) : 0.7673
  AUC-ROC  = 0.9209
  AUC-PR   = 0.3574
  F2       = 0.1809   F1 = 0.2242   Precision = 0.3731   Recall = 0.1603


---
## 6. Cross-Validation — TimeSeriesSplit
Sampling is wrapped inside `imblearn.Pipeline` so it fires only on training folds, never on validation folds.

In [40]:
train_sorted = train_df.sort_values("time_6h").reset_index(drop=True)
X_cv_raw, _  = prepare_features(train_sorted)
for c in feature_cols:
    if c not in X_cv_raw.columns:
        X_cv_raw[c] = 0
X_cv = X_cv_raw[feature_cols].fillna(0).values
y_cv = train_sorted[TARGET].values

tscv = TimeSeriesSplit(n_splits=5)

cv_pipelines = {
    "Baseline"          : ImbPipeline([("clf", XGBClassifier(**BASE_XGB_PARAMS))]),
    "RandomOverSampler" : ImbPipeline([("smp", RandomOverSampler(random_state=42)),
                                       ("clf", XGBClassifier(**BASE_XGB_PARAMS))]),
    "RandomUnderSampler": ImbPipeline([("smp", RandomUnderSampler(random_state=42)),
                                       ("clf", XGBClassifier(**BASE_XGB_PARAMS))]),
    "SMOTE"             : ImbPipeline([("smp", SMOTE(random_state=42, k_neighbors=5)),
                                       ("clf", XGBClassifier(**BASE_XGB_PARAMS))]),
    "TomekLinks"        : ImbPipeline([("smp", TomekLinks(n_jobs=-1)),
                                       ("clf", XGBClassifier(**BASE_XGB_PARAMS))]),
    "SMOTETomek"        : ImbPipeline([("smp", SMOTETomek(random_state=42)),
                                       ("clf", XGBClassifier(**BASE_XGB_PARAMS))]),
}
if "ADASYN" in all_results:
    cv_pipelines["ADASYN"] = ImbPipeline([("smp", ADASYN(random_state=42)),
                                           ("clf", XGBClassifier(**BASE_XGB_PARAMS))])

print(f"CV: TimeSeriesSplit(n_splits=5)   scoring=average_precision")
print(f"Train size: {X_cv.shape[0]:,} rows\n")

cv_results = {}
for name, pipe in cv_pipelines.items():
    try:
        scores = cross_val_score(
            pipe, X_cv, y_cv,
            cv=tscv, scoring="average_precision", n_jobs=-1,
        )
        valid = scores[~np.isnan(scores)]
        cv_results[name] = {
            "mean" : valid.mean() if len(valid) else float("nan"),
            "std"  : valid.std()  if len(valid) else float("nan"),
            "folds": scores,
        }
        print(f"{name:<22s}: AUC-PR = {cv_results[name]['mean']:.4f} "
              f"± {cv_results[name]['std']:.4f}   folds = {np.round(scores, 4)}")
    except Exception as exc:
        print(f"{name:<22s}: FAILED — {exc}")

CV: TimeSeriesSplit(n_splits=5)   scoring=average_precision
Train size: 22,098 rows

Baseline              : AUC-PR = 0.2147 ± 0.1395   folds = [   nan 0.2177 0.44   0.1224 0.0785]
RandomOverSampler     : AUC-PR = 0.2087 ± 0.1954   folds = [   nan 0.0894 0.5444 0.1346 0.0663]
RandomUnderSampler    : AUC-PR = 0.1654 ± 0.1567   folds = [   nan 0.0564 0.11   0.4344 0.0607]
SMOTE                 : AUC-PR = 0.2023 ± 0.1965   folds = [   nan 0.0205 0.5295 0.173  0.0862]
TomekLinks            : AUC-PR = 0.2159 ± 0.1458   folds = [   nan 0.2088 0.4533 0.1337 0.0678]
SMOTETomek            : AUC-PR = 0.2039 ± 0.1964   folds = [   nan 0.0205 0.5295 0.1809 0.0849]
ADASYN                : AUC-PR = 0.2044 ± 0.1997   folds = [   nan 0.0207 0.537  0.1752 0.0846]


---
## 7. Results Table

In [41]:
results_df = pd.DataFrame(list(all_results.values())).set_index("Strategy")
results_df["CV AUC-PR"] = [
    round(cv_results.get(k, {}).get("mean", float("nan")), 4)
    for k in results_df.index
]
results_df = results_df.sort_values("Recall", ascending=False)

display(
    results_df.style
    .highlight_max(subset=["AUC-ROC", "AUC-PR", "F2", "Recall"], color="#b7e4c7")
    .highlight_min(subset=["F2", "Recall"],                       color="#ffb3b3")
    .format({
        "AUC-ROC"  : "{:.4f}",
        "AUC-PR"   : "{:.4f}",
        "F1"       : "{:.4f}",
        "F2"       : "{:.4f}",
        "Precision": "{:.4f}",
        "Recall"   : "{:.4f}",
        "Threshold": "{:.4f}",
        "CV AUC-PR": "{:.4f}",
    })
    .set_caption("Sorted by Recall. Green = best, Red = worst. F2 = recall-weighted metric.")
)

,AUC-ROC,AUC-PR,F1,F2,Precision,Recall,Threshold,CV AUC-PR
Strategy,,,,,,,,
RandomOverSampler,0.9231,0.4480,0.4633,0.4883,0.4270,0.5064,0.1739,0.2087
TomekLinks,0.9329,0.4228,0.4470,0.4774,0.4041,0.5000,0.0494,0.2159
Baseline,0.9314,0.4004,0.3460,0.2908,0.5062,0.2628,0.2190,0.2147
ADASYN,0.9271,0.3637,0.2477,0.1968,0.4355,0.1731,0.6730,0.2044
SMOTE,0.9238,0.3617,0.2298,0.1920,0.3418,0.1731,0.5657,0.2023
RandomUnderSampler,0.9128,0.3106,0.2381,0.1844,0.4630,0.1603,0.9670,0.1654
SMOTETomek,0.9209,0.3574,0.2242,0.1809,0.3731,0.1603,0.7673,0.2039


---
## 8. RandomOverSampler — Selection Rationale

**Manually selected: RandomOverSampler**

- **Threshold is now tuned by maximizing F2-score** (beta=2) on the val window with `min_precision ≥ 5%`
  — this directly pushes the threshold lower to catch more floods instead of the previous F1-optimal threshold which collapsed to 0.9994 (missing 94% of floods)
- ROS creates a balanced 50/50 fit set, giving the model strong flood signal without destroying the majority class structure
- Among oversampling strategies, ROS + F2 threshold achieves the highest **Recall** and **F2** on the test set
- Missing a real flood is catastrophically more costly than a false alarm → Recall must be the primary metric

In [42]:
# Hardcoded selection — RandomOverSampler
SELECTED = "RandomOverSampler"
sel_model, sel_threshold = saved_models[SELECTED]
sel_metrics = all_results[SELECTED]
sel_probs   = all_probs[SELECTED]
sel_preds   = (sel_probs >= sel_threshold).astype(int)

print(f"Selected strategy : {SELECTED}")
print(f"Threshold         : {sel_threshold:.4f}")
print(f"AUC-ROC           : {sel_metrics['AUC-ROC']:.4f}")
print(f"AUC-PR            : {sel_metrics['AUC-PR']:.4f}")
print(f"F1                : {sel_metrics['F1']:.4f}")
print(f"Precision         : {sel_metrics['Precision']:.4f}")
print(f"Recall            : {sel_metrics['Recall']:.4f}")
print()
print(classification_report(y_test, sel_preds,
                             target_names=["No Flood", "Flood"],
                             zero_division=0))

Selected strategy : RandomOverSampler
Threshold         : 0.1739
AUC-ROC           : 0.9231
AUC-PR            : 0.4480
F1                : 0.4633
Precision         : 0.4270
Recall            : 0.5064

              precision    recall  f1-score   support

    No Flood       0.99      0.98      0.98      5370
       Flood       0.43      0.51      0.46       156

    accuracy                           0.97      5526
   macro avg       0.71      0.74      0.72      5526
weighted avg       0.97      0.97      0.97      5526



---
## 9. Hyperparameter Optimization — Optuna

**Objective:** optimize the RandomOverSampler + XGBoost pipeline using Optuna TPE sampler.

**Strategy:**
- Each trial runs `TimeSeriesSplit(n_splits=5)` internally
- ROS is applied only to each training fold (never to the validation fold)
- Folds with zero positive examples (first fold issue) are skipped; remaining folds are averaged
- **Maximize: F2-score** (beta=2 — recall is weighted 4× more than precision, aligned with flood detection priority)
- Trials: 60

In [43]:
optuna.logging.set_verbosity(optuna.logging.WARNING)

tscv_opt = TimeSeriesSplit(n_splits=5)


def objective(trial):
    params = {
        "n_estimators"    : trial.suggest_int("n_estimators",      100, 900),
        "learning_rate"   : trial.suggest_float("learning_rate",   0.005, 0.3,  log=True),
        "max_depth"       : trial.suggest_int("max_depth",         3, 10),
        "min_child_weight": trial.suggest_int("min_child_weight",  1, 15),
        "subsample"       : trial.suggest_float("subsample",       0.4, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree",0.4, 1.0),
        "reg_alpha"       : trial.suggest_float("reg_alpha",       1e-8, 20.0, log=True),
        "reg_lambda"      : trial.suggest_float("reg_lambda",      1e-8, 20.0, log=True),
        "gamma"           : trial.suggest_float("gamma",           0.0, 5.0),
        "eval_metric"     : "aucpr",
        "random_state"    : 42,
        "n_jobs"          : -1,
    }

    fold_scores = []
    for fold_tr_idx, fold_val_idx in tscv_opt.split(X_cv):
        X_f_tr,  y_f_tr  = X_cv[fold_tr_idx],  y_cv[fold_tr_idx]
        X_f_val, y_f_val = X_cv[fold_val_idx], y_cv[fold_val_idx]

        # Skip folds with no positive examples in train or val
        if y_f_tr.sum() < 2 or y_f_val.sum() == 0:
            continue

        X_res, y_res = RandomOverSampler(random_state=42).fit_resample(X_f_tr, y_f_tr)

        mdl = XGBClassifier(**params)
        mdl.fit(X_res, y_res, verbose=False)

        probs = mdl.predict_proba(X_f_val)[:, 1]

        # F2-optimal threshold per fold — aligned with recall priority
        precision_c, recall_c, thresh_c = precision_recall_curve(y_f_val, probs)
        fbeta_c = ((1 + 4) * precision_c * recall_c
                   / (4 * precision_c + recall_c + 1e-9))
        valid_mask = precision_c[:-1] >= 0.05
        if valid_mask.any():
            best_thr = thresh_c[np.argmax(np.where(valid_mask, fbeta_c[:-1], -1.0))]
        else:
            best_thr = thresh_c[np.argmax(fbeta_c[:-1])]

        preds = (probs >= best_thr).astype(int)
        fold_scores.append(fbeta_score(y_f_val, preds, beta=2, zero_division=0))

    return float(np.mean(fold_scores)) if fold_scores else 0.0


study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=42),
)
study.optimize(objective, n_trials=60, show_progress_bar=True)

print(f"\nOptimization complete.")
print(f"Best CV F2     : {study.best_value:.4f}")
print(f"Best params    :")
for k, v in study.best_params.items():
    print(f"  {k:<22s}: {v}")

Best trial: 57. Best value: 0.491207: 100%|██████████| 60/60 [01:26<00:00,  1.45s/it]


Optimization complete.
Best CV F2     : 0.4912
Best params    :
  n_estimators          : 221
  learning_rate         : 0.08990156202701051
  max_depth             : 5
  min_child_weight      : 3
  subsample             : 0.8818791811269663
  colsample_bytree      : 0.4177772720303967
  reg_alpha             : 18.559220438224635
  reg_lambda            : 0.00010898227110634987
  gamma                 : 1.2408089758851817


In [44]:
# Trial history — convergence to best F2 value
trial_values = [t.value for t in study.trials if t.value is not None]
best_so_far  = [max(trial_values[:i+1]) for i in range(len(trial_values))]

print("Trial progression (CV F2-score):")
for i, (tv, bsf) in enumerate(zip(trial_values, best_so_far)):
    marker = " <-- new best" if tv == bsf and (i == 0 or bsf > best_so_far[i-1]) else ""
    print(f"  Trial {i+1:>3d}: {tv:.4f}  (best so far: {bsf:.4f}){marker}")

Trial progression (CV F2-score):
  Trial   1: 0.4663  (best so far: 0.4663) <-- new best
  Trial   2: 0.4180  (best so far: 0.4663)
  Trial   3: 0.4249  (best so far: 0.4663)
  Trial   4: 0.4720  (best so far: 0.4720) <-- new best
  Trial   5: 0.4398  (best so far: 0.4720)
  Trial   6: 0.4347  (best so far: 0.4720)
  Trial   7: 0.4148  (best so far: 0.4720)
  Trial   8: 0.4156  (best so far: 0.4720)
  Trial   9: 0.4321  (best so far: 0.4720)
  Trial  10: 0.3990  (best so far: 0.4720)
  Trial  11: 0.4462  (best so far: 0.4720)
  Trial  12: 0.4353  (best so far: 0.4720)
  Trial  13: 0.4316  (best so far: 0.4720)
  Trial  14: 0.4748  (best so far: 0.4748) <-- new best
  Trial  15: 0.4474  (best so far: 0.4748)
  Trial  16: 0.4281  (best so far: 0.4748)
  Trial  17: 0.4064  (best so far: 0.4748)
  Trial  18: 0.4843  (best so far: 0.4843) <-- new best
  Trial  19: 0.4375  (best so far: 0.4843)
  Trial  20: 0.4897  (best so far: 0.4897) <-- new best
  Trial  21: 0.4297  (best so far: 0.4897)

---
## 10. Tuned Model — Final Training & Evaluation

In [45]:
# Attach fixed fields to best params from Optuna
best_params = {
    **study.best_params,
    "eval_metric" : "aucpr",
    "random_state": 42,
    "n_jobs"      : -1,
}

# Resample fit set with ROS, then train with tuned params
X_ros, y_ros = RandomOverSampler(random_state=42).fit_resample(X_tr, y_tr)
print(f"Fit rows (resampled) : {X_ros.shape[0]:,}   flood rate = {y_ros.mean()*100:.1f}%")

tuned_model = XGBClassifier(**best_params)
tuned_model.fit(X_ros, y_ros, eval_set=[(X_val, y_val)], verbose=False)

# F2-optimal threshold on validation window
tuned_threshold        = best_threshold_for_recall(tuned_model, X_val, y_val)
tuned_metrics, tuned_probs = evaluate(tuned_model, X_test, y_test,
                                       tuned_threshold, "ROS + Optuna")
tuned_preds = (tuned_probs >= tuned_threshold).astype(int)

print(f"\nThreshold (F2-tuned) : {tuned_threshold:.4f}")
print(f"AUC-ROC   : {tuned_metrics['AUC-ROC']:.4f}")
print(f"AUC-PR    : {tuned_metrics['AUC-PR']:.4f}")
print(f"F2        : {tuned_metrics['F2']:.4f}")
print(f"F1        : {tuned_metrics['F1']:.4f}")
print(f"Precision : {tuned_metrics['Precision']:.4f}")
print(f"Recall    : {tuned_metrics['Recall']:.4f}")

Fit rows (resampled) : 37,322   flood rate = 50.0%

Threshold (F2-tuned) : 0.3865
AUC-ROC   : 0.9294
AUC-PR    : 0.3976
F2        : 0.4771
F1        : 0.4389
Precision : 0.3873
Recall    : 0.5064


In [46]:
# Base ROS vs Optuna-tuned ROS — side-by-side comparison
base_m = sel_metrics

compare = pd.DataFrame([
    {"Model"    : "ROS  (base params)",
     "AUC-ROC" : base_m["AUC-ROC"],
     "AUC-PR"  : base_m["AUC-PR"],
     "F2"      : base_m["F2"],
     "F1"      : base_m["F1"],
     "Precision": base_m["Precision"],
     "Recall"  : base_m["Recall"],
     "Threshold": base_m["Threshold"]},
    {"Model"    : "ROS  (Optuna tuned)",
     "AUC-ROC" : tuned_metrics["AUC-ROC"],
     "AUC-PR"  : tuned_metrics["AUC-PR"],
     "F2"      : tuned_metrics["F2"],
     "F1"      : tuned_metrics["F1"],
     "Precision": tuned_metrics["Precision"],
     "Recall"  : tuned_metrics["Recall"],
     "Threshold": tuned_metrics["Threshold"]},
]).set_index("Model")

# Delta row
delta = compare.loc["ROS  (Optuna tuned)"] - compare.loc["ROS  (base params)"]
delta.name = "Delta (tuned - base)"
compare = pd.concat([compare, delta.to_frame().T])

display(
    compare.style
    .highlight_max(subset=["AUC-PR", "F2", "Recall"], color="#b7e4c7")
    .format("{:.4f}")
    .set_caption("Base ROS vs Optuna-tuned ROS — test set")
)

print("\nClassification Report — Optuna Tuned Model:")
print(classification_report(y_test, tuned_preds,
                             target_names=["No Flood", "Flood"],
                             zero_division=0))

,AUC-ROC,AUC-PR,F2,F1,Precision,Recall,Threshold
ROS (base params),0.9231,0.4480,0.4883,0.4633,0.4270,0.5064,0.1739
ROS (Optuna tuned),0.9294,0.3976,0.4771,0.4389,0.3873,0.5064,0.3865
Delta (tuned - base),0.0063,-0.0504,-0.0112,-0.0244,-0.0397,0.0000,0.2126



Classification Report — Optuna Tuned Model:
              precision    recall  f1-score   support

    No Flood       0.99      0.98      0.98      5370
       Flood       0.39      0.51      0.44       156

    accuracy                           0.96      5526
   macro avg       0.69      0.74      0.71      5526
weighted avg       0.97      0.96      0.97      5526



---
## 11. Final Report & Save

In [47]:
SEP = "=" * 62
print(SEP)
print("  BAKU SENTINEL — Day 05 Optimization Report")
print(SEP)
print(f"  Data range : {config.HISTORICAL_START} → {config.HISTORICAL_END}")
print(f"  Gold rows  : {len(df):,}  |  Flood rate: {df[TARGET].mean()*100:.2f}%")
print(f"  Features   : {len(feature_cols)}")
print(f"  Optuna     : 60 trials  |  Best CV F2: {study.best_value:.4f}")
print(f"  Threshold method: F2-optimal (beta=2, min_precision=5%)")
print()
print("  STRATEGY COMPARISON (test set, sorted by Recall)")
for name, row in results_df.iterrows():
    flag = " <-- SELECTED" if name == SELECTED else ""
    print(f"  {name:<22s}  F2={row['F2']:.4f}  "
          f"Recall={row['Recall']:.4f}  Precision={row['Precision']:.4f}{flag}")
print()
print("  BASE ROS vs OPTUNA-TUNED ROS")
for col in ["AUC-ROC", "AUC-PR", "F2", "F1", "Precision", "Recall", "Threshold"]:
    b = base_m[col]
    t = tuned_metrics[col]
    arrow = "up" if t > b else ("down" if t < b else "=")
    print(f"  {col:<12s}: base={b:.4f}  tuned={t:.4f}  [{arrow}] {abs(t-b):.4f}")
print(SEP)

  BAKU SENTINEL — Day 05 Optimization Report
  Data range : 2020-01-01 → 2026-04-20
  Gold rows  : 27,624  |  Flood rate: 1.06%
  Features   : 37
  Optuna     : 60 trials  |  Best CV F2: 0.4912
  Threshold method: F2-optimal (beta=2, min_precision=5%)

  STRATEGY COMPARISON (test set, sorted by Recall)
  RandomOverSampler       F2=0.4883  Recall=0.5064  Precision=0.4270 <-- SELECTED
  TomekLinks              F2=0.4774  Recall=0.5000  Precision=0.4041
  Baseline                F2=0.2908  Recall=0.2628  Precision=0.5062
  ADASYN                  F2=0.1968  Recall=0.1731  Precision=0.4355
  SMOTE                   F2=0.1920  Recall=0.1731  Precision=0.3418
  RandomUnderSampler      F2=0.1844  Recall=0.1603  Precision=0.4630
  SMOTETomek              F2=0.1809  Recall=0.1603  Precision=0.3731

  BASE ROS vs OPTUNA-TUNED ROS
  AUC-ROC     : base=0.9231  tuned=0.9294  [up] 0.0063
  AUC-PR      : base=0.4480  tuned=0.3976  [down] 0.0504
  F2          : base=0.4883  tuned=0.4771  [down] 0.0112

In [48]:
# Save tuned model and metadata
save_path = config.MODELS_DIR / "day05_ros_optuna.joblib"
joblib.dump(
    {"model"    : tuned_model,
     "features" : feature_cols,
     "threshold": tuned_threshold,
     "strategy" : "RandomOverSampler + Optuna",
     "params"   : best_params},
    save_path,
)

meta = {
    **tuned_metrics,
    "strategy"       : "RandomOverSampler + Optuna",
    "optuna_trials"  : 60,
    "optuna_best_cv" : round(study.best_value, 4),
    "best_params"    : best_params,
}
meta_path = config.MODELS_DIR / "day05_ros_optuna_metrics.json"
with open(meta_path, "w") as f:
    json.dump(meta, f, indent=2)

print(f"Model : {save_path}")
print(f"Meta  : {meta_path}")

Model : /home/aliagabalayev/Desktop/Workspace/Weather-Prediction/models/day05_ros_optuna.joblib
Meta  : /home/aliagabalayev/Desktop/Workspace/Weather-Prediction/models/day05_ros_optuna_metrics.json
